# TRL GRPOTrainer single-turn verifiable reward

**Objective:** single-turn deterministic scoring; study reward ties, advantage and clipping; do NOT represent as Harbor multi-turn RL.

This is a guided training starter. Training is deliberately **not executed** until you complete the numbered checks below. Prerequisite: [GRPO mechanics lab](01_grpo_clipping_diagnostics.ipynb). Save the checkpoint and metric provenance for the next lab.


# Learning companion: 03 grpotrainer verifiable

**Where this fits:** This lab follows the connected APT curriculum and trains one research skill rather than just producing code. Its student exercises below remain unfinished on purpose. Read the explanation and worked example before implementing anything.

### Why this problem matters

GRPO optimizes reward-graded model generations, not curated reference completions.

### Concepts and definitions

Sample several completions per prompt, grade them with a reliable reward, form group-relative advantages, then update the policy using token-level objectives and clipping.

**Core mathematical object**

$$A_i=\frac{R_i-\bar R}{s_R+\delta},\qquad r_{i,t}=\exp(\ell_{\theta,i,t}-\ell_{\mathrm{old},i,t})$$

Interpret every quantity with its denominator, conditioning context, measurement source and train/eval boundary; a formula is not evidence until those details are fixed.

### Worked example (separate from your exercise)

Suppose a group receives rewards [1,0,1,0]. With mean 0.5 the group-centered signals have signs [+,-,+,-]. With rewards [1,1,1,1] there is no within-group reward discrimination.

**Reason through the example:** Which conclusion is justified directly by the given observations? Which alternative explanation remains possible? What extra measurement would discriminate between them? This is the habit that the APT project is trying to build.

### Guided code orientation and independent lab

Before writing code, identify the input record fields, the output artifact and the invariants the transformation must preserve. If the supplied scaffold imports TRL or another framework, inspect the runtime version and print the relevant API signature before running training.

**Your implementation assignment:** Implement an exact-answer verifier for one toy prompt; check it rejects unintended answers and inspect group reward variance before enabling training.

Use the existing notebook TODOs as your working area. You own the core transformation, metric arithmetic, split logic and tests. The notebook provides setup and rendering scaffolds rather than filling these steps for you.

### Diagnostic graph assignment

Reward ties, advantage spread, clipping/KL and independent held-out success; a toy answer checker is not a Harbor verifier.

For each graph specify: x/y meaning and units, denominator, aggregation window, missing values, uncertainty, source and checkpoint. Calculate the plotting arrays yourself, then use the existing publication-quality render helper or adapt its labels and export settings.

### Interpretation and transfer

Write (1) measured observation, (2) plausible competing hypotheses, (3) confounder or leakage check, (4) discriminating next experiment, and (5) saved artifact consumed by the next lab. Do not interpret training loss or synthetic examples as trusted task success.

In [ ]:
# Colab GPU recommended. Validate TRL version and tokenizer/model requirements.
%pip -q install -U "trl" "transformers" "datasets" "accelerate" "matplotlib"

import json, inspect
import trl
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
print("TRL", trl.__version__)
print("Trainer constructor:", inspect.signature(GRPOTrainer))
print("Config constructor:", inspect.signature(GRPOConfig))

# TODO(student): choose a small compatible causal LM/checkpoint based on actual GPU memory.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"


In [ ]:
# Intentionally verifiable SINGLE-TURN toy reward, not Harbor agent RL.
dataset = Dataset.from_list([
    {"prompt": [{"role": "user", "content": "Return exactly the word blue."}]},
    {"prompt": [{"role": "user", "content": "Return exactly the word green."}]},
])
def exact_color_reward(completions, **kwargs):
    # TODO(student): verify this signature against installed TRL and reward test fixtures.
    allowed = {"blue", "green"}
    texts = [c[0]["content"] if isinstance(c, list) else str(c) for c in completions]
    return [1.0 if t.strip().lower() in allowed else 0.0 for t in texts]

config = GRPOConfig(
    output_dir="artifacts/grpo-single-turn",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=64,
    max_prompt_length=128,
    logging_steps=1,
    report_to="none",
)
# TODO(student): confirm effective generation batch >= num_generations, reward-group diversity,
# and the exact semantics of sampled-completion reward for your task.
# trainer = GRPOTrainer(model=MODEL_ID, args=config, train_dataset=dataset,
#                       reward_funcs=exact_color_reward)
# trainer.train()
# TODO(student): collect ratio clipping, KL, gradient norms, reward ties and trusted eval.


## Diagnostics and plotting assignment
1. Define the numerical quantities and denominators. 2. Extract training/eval measurements. 3. Compare the appropriate checkpoints and controlled baselines. 4. Interpret discrepancies and design next discriminating probe.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_metric(x, y, title, ylabel, path):
    """Rendering ONLY. Student must implement x/y calculation from real trainer logs."""
    fig, ax = plt.subplots(figsize=(7.2, 4.5), layout="constrained")
    ax.plot(x, y, marker=".", linewidth=1.8)
    ax.set(title=title, xlabel="optimizer step", ylabel=ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()

# TODO(student): extract metric points with provenance from trainer.state.log_history.
# TODO(student): choose correct step axis, smoothing policy, and diagnostic comparisons.
# plot_metric(steps, values, "Training diagnostic", "Value", "artifacts/diagnostic.png")


## Exit criteria
- Record model revision, TRL version, tokenizer, seed and GPU.
- Confirm split integrity, verifier correctness, and no label/answer leakage.
- Plot at least three diagnostics with student-computed axes and quantities.
- Save an experiment report with measured outcomes versus hypotheses.
- Do not claim Harbor multi-turn GRPO integration from this notebook.
